# 01 · Bronze (local + pandas) — escrevendo no MinIO

```
data/lote1 ──► landing (bucket MinIO) ──► BRONZE (Parquet no bucket warehouse)
```

Aqui **não há Spark**: o pandas lê os arquivos da landing no MinIO, acrescenta as 3 colunas de rastreio (`_ingest_ts`, `_source_file`, `_batch_id`) e grava **Parquet** no bucket `warehouse`.

**Diferença para a versão Spark:** estas são pastas de arquivos Parquet por lote, **não tabelas Iceberg**. Iceberg precisa de um motor como o Spark para escrever os metadados. A ideia da Bronze é a mesma: guardar como veio, com rastreabilidade, e poder reexecutar sem duplicar.

**Antes de começar:** o MinIO precisa estar no ar (`docker compose up -d`) e o ambiente precisa ter `pandas`, `pyarrow` e `boto3`.

## Passo 0 · Conexão com o MinIO e criação dos buckets

In [1]:
import io
import os
from datetime import datetime, timezone
from pathlib import Path

import boto3
import pandas as pd

ENDPOINT = os.environ.get("MINIO_ENDPOINT", "http://localhost:9000")
USUARIO = os.environ.get("MINIO_USER", "admin")
SENHA = os.environ.get("MINIO_PASSWORD", "admin12345")
BATCH = "lote1"
PREFIXO_LANDING = f"mercadonordeste/{BATCH}"

# a pasta data pode estar ao lado do notebook ou uma pasta acima
DATA = next(p for p in (Path("data"), Path("../data")) if (p / BATCH).exists()) / BATCH

s3 = boto3.client("s3", endpoint_url=ENDPOINT, aws_access_key_id=USUARIO,
                  aws_secret_access_key=SENHA, region_name="us-east-1")

existentes = {b["Name"] for b in s3.list_buckets()["Buckets"]}
for bucket in ("landing", "warehouse"):
    if bucket not in existentes:
        s3.create_bucket(Bucket=bucket)
print("buckets:", sorted(b["Name"] for b in s3.list_buckets()["Buckets"]))

buckets: ['landing', 'warehouse']


## Passo 1 · Os arquivos chegam na landing
Enviamos os 4 arquivos como um sistema de origem faria, pela API S3. A landing guarda o dado **exatamente como veio**.

In [2]:
for arquivo in sorted(DATA.iterdir()):
    s3.upload_file(str(arquivo), "landing", f"{PREFIXO_LANDING}/{arquivo.name}")

for o in s3.list_objects_v2(Bucket="landing", Prefix=PREFIXO_LANDING)["Contents"]:
    print(f"{o['Size']:>10,} bytes  s3://landing/{o['Key']}")

   529,223 bytes  s3://landing/mercadonordeste/lote1/avaliacoes.csv
   420,891 bytes  s3://landing/mercadonordeste/lote1/clientes.csv
 7,601,232 bytes  s3://landing/mercadonordeste/lote1/pedidos.jsonl
     9,044 bytes  s3://landing/mercadonordeste/lote1/produtos.csv


Abra o console do MinIO em <http://localhost:9001> (usuário e senha do seu `.env`) e veja o bucket **landing**.
É um *object storage*: não há pastas de verdade, só chaves com `/` no nome.

### Experimento: o que é uma "pasta" aqui?

Vamos pedir ao MinIO as chaves de três formas e fazer o experimento de **renomear**.
Antes de rodar, pense: se não existem pastas, como o console mostra `mercadonordeste` e `lote1` como pastas?

In [3]:
# função auxiliar: lista (chave, tamanho) de todos os objetos de um prefixo (com paginação)
def listar(bucket, prefixo=""):
    achados = []
    for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefixo):
        achados += [(o["Key"], o["Size"]) for o in pagina.get("Contents", [])]
    return achados


# 1) "Pastas" são só prefixos: pedimos ao MinIO para agrupar as chaves pelo "/"
resp = s3.list_objects_v2(Bucket="landing", Prefix="mercadonordeste/", Delimiter="/")
print("Prefixos:", [p["Prefix"] for p in resp.get("CommonPrefixes", [])])

# 2) Listar por prefixo
for chave, tam in listar("landing", "mercadonordeste/lote1/"):
    print(f"{tam:>10,} bytes  {chave}")

# 3) Renomear não existe: é COPIAR e APAGAR (numa chave de teste, sem mexer nos dados)
s3.put_object(Bucket="landing", Key="teste/a.txt", Body=b"ola")
s3.copy_object(Bucket="landing", Key="teste/b.txt",
               CopySource={"Bucket": "landing", "Key": "teste/a.txt"})
s3.delete_object(Bucket="landing", Key="teste/a.txt")
print("Depois do 'renomear':", [c for c, _ in listar("landing", "teste/")])
s3.delete_object(Bucket="landing", Key="teste/b.txt")      # limpa o teste

Prefixos: ['mercadonordeste/lote1/']
   529,223 bytes  mercadonordeste/lote1/avaliacoes.csv
   420,891 bytes  mercadonordeste/lote1/clientes.csv
 7,601,232 bytes  mercadonordeste/lote1/pedidos.jsonl
     9,044 bytes  mercadonordeste/lote1/produtos.csv
Depois do 'renomear': ['teste/b.txt']


{'ResponseMetadata': {'RequestId': '18E21DBF789FE57B',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18E21DBF789FE57B',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '8481',
   'x-ratelimit-remaining': '8481',
   'x-xss-protection': '1; mode=block',
   'date': 'Mon, 26 Oct 2026 15:25:09 GMT'},
  'RetryAttempts': 0}}

A "pasta" é só um **prefixo**: o MinIO a simula quando pedimos o agrupamento por `/`. Renomear são duas operações (copiar e apagar). Num bucket com milhões de objetos, "renomear uma pasta" significa copiar tudo.

## Passo 2 · Landing → Bronze
- Os CSVs são lidos como **texto** (`dtype=str`), sem tipar. Tipar é trabalho da Prata.
- O JSON é lido sem inferência de tipos (`dtype=False`) e `itens` continua sendo uma lista.
- **Idempotência:** antes de gravar, apagamos o que este lote já gravou. Reexecutar não duplica.

In [4]:
INGEST_TS = datetime.now(timezone.utc)


def ler_landing(nome):
    return s3.get_object(Bucket="landing", Key=f"{PREFIXO_LANDING}/{nome}")["Body"].read()


def com_rastreio(df, nome_arquivo):
    df = df.copy()
    df["_ingest_ts"] = INGEST_TS
    df["_source_file"] = f"s3://landing/{PREFIXO_LANDING}/{nome_arquivo}"
    df["_batch_id"] = BATCH
    return df


def apagar_prefixo(bucket, prefixo):
    for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefixo):
        chaves = [{"Key": o["Key"]} for o in pagina.get("Contents", [])]
        if chaves:
            s3.delete_objects(Bucket=bucket, Delete={"Objects": chaves})


def gravar_bronze(df, tabela):
    prefixo = f"bronze/{tabela}/{BATCH}/"
    apagar_prefixo("warehouse", prefixo)          # idempotência: refaz só este lote
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    s3.put_object(Bucket="warehouse", Key=f"{prefixo}part-0000.parquet", Body=buffer.getvalue())
    return len(df)


def ingerir_lote():
    csv = lambda nome: pd.read_csv(io.BytesIO(ler_landing(nome)), dtype=str)
    pedidos = pd.read_json(io.BytesIO(ler_landing("pedidos.jsonl")), lines=True,
                           dtype=False, convert_dates=False)
    fontes = {
        "clientes":   (csv("clientes.csv"), "clientes.csv"),
        "produtos":   (csv("produtos.csv"), "produtos.csv"),
        "pedidos":    (pedidos, "pedidos.jsonl"),
        "avaliacoes": (csv("avaliacoes.csv"), "avaliacoes.csv"),
    }
    return {tabela: gravar_bronze(com_rastreio(df, arq), tabela) for tabela, (df, arq) in fontes.items()}


ingerir_lote()

{'clientes': 5000, 'produtos': 200, 'pedidos': 30150, 'avaliacoes': 6489}

Resultado esperado, aproximadamente: **clientes 5.000 · produtos 200 · pedidos 30.150 · avaliações 6.489**. Os pedidos incluem duplicados de propósito, e a Bronze **não** os remove.

## Passo 3 · Ler de volta e inspecionar

In [5]:
def ler_bronze(tabela):
    partes = []
    for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket="warehouse", Prefix=f"bronze/{tabela}/"):
        for o in pagina.get("Contents", []):
            corpo = s3.get_object(Bucket="warehouse", Key=o["Key"])["Body"].read()
            partes.append(pd.read_parquet(io.BytesIO(corpo)))
    return pd.concat(partes, ignore_index=True)


pedidos = ler_bronze("pedidos")
print(pedidos.shape)
print(pedidos.dtypes)
pedidos[["pedido_id", "cliente_id", "status", "itens", "_batch_id", "_source_file"]].head(3)

(30150, 10)
pedido_id                        int64
cliente_id                       int64
data_pedido                        str
status                             str
forma_pagamento                    str
valor_frete                    float64
itens                           object
_ingest_ts         datetime64[us, UTC]
_source_file                       str
_batch_id                          str
dtype: object


,pedido_id,cliente_id,status,itens,_batch_id,_source_file
0,123912,2769,devolvido,"[{'produto_id': 6, 'qtd': 2, 'preco_unit': 211...",lote1,s3://landing/mercadonordeste/lote1/pedidos.jsonl
1,106631,3227,enviado,"[{'produto_id': 39, 'qtd': 1, 'preco_unit': 82...",lote1,s3://landing/mercadonordeste/lote1/pedidos.jsonl
2,127545,4575,entregue,"[{'produto_id': 82, 'qtd': 3, 'preco_unit': 22...",lote1,s3://landing/mercadonordeste/lote1/pedidos.jsonl


Repare: `itens` continua sendo uma lista, e as colunas dos CSVs são texto. Isso é a Bronze.

## Passo 4 · Onde está a tabela `pedidos`?

Aqui não existe catálogo: a "tabela" é só uma convenção de **prefixo**, `bronze/<tabela>/<lote>/`, com um Parquet dentro. Vamos olhar o que o storage guarda de verdade.

In [6]:
def resumo_objetos(prefixo):
    objs = listar("warehouse", prefixo)
    return {"objetos": len(objs), "KB": round(sum(t for _, t in objs) / 1024)}


for chave, tam in listar("warehouse", "bronze/"):
    print(f"{tam:>10,}  s3://warehouse/{chave}")
print(resumo_objetos("bronze/pedidos/"))

   137,334  s3://warehouse/bronze/avaliacoes/lote1/part-0000.parquet
   120,390  s3://warehouse/bronze/clientes/lote1/part-0000.parquet
   770,513  s3://warehouse/bronze/pedidos/lote1/part-0000.parquet
    10,410  s3://warehouse/bronze/produtos/lote1/part-0000.parquet
{'objetos': 1, 'KB': 752}


Confira também no console do MinIO (<http://localhost:9001>), no bucket **warehouse**, pasta `bronze`.

**Para discutir:** não há `metadata/`. Nada diz quais arquivos fazem parte da tabela além do nome do prefixo. Se alguém colocar um arquivo qualquer em `bronze/pedidos/lote1/`, a função `ler_bronze` vai lê-lo como se fosse dado. Na versão Spark/Iceberg, a pasta `metadata/` registra exatamente quais arquivos formam cada versão da tabela.

## Passo 5 · Idempotência
Rode a ingestão de novo. A contagem de linhas **não** dobra. Vamos conferir também o que aconteceu com os **objetos**.

In [8]:
def objetos(prefixo):
    achados = {}
    for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket="warehouse", Prefix=prefixo):
        for o in pagina.get("Contents", []):
            achados[o["Key"]] = o["LastModified"]
    return achados


prefixo = "bronze/pedidos/"
linhas_antes, obj_antes = len(ler_bronze("pedidos")), objetos(prefixo)

ingerir_lote()                                    # reexecuta o MESMO lote

linhas_depois, obj_depois = len(ler_bronze("pedidos")), objetos(prefixo)

print("Linhas :", linhas_antes, "->", linhas_depois, "| idempotente:", linhas_antes == linhas_depois)
print("Objetos:", len(obj_antes), "->", len(obj_depois))
for chave in obj_depois:
    print(f"  {chave}\n    gravado em {obj_antes.get(chave)}  ->  {obj_depois[chave]}")

Linhas : 30150 -> 30150 | idempotente: True
Objetos: 1 -> 1
  bronze/pedidos/lote1/part-0000.parquet
    gravado em 2026-10-26 15:25:11.756000+00:00  ->  2026-10-26 15:40:46.063000+00:00


### O que aconteceu?

As **linhas** não dobraram e o **número de objetos** também não mudou, mas a data de gravação do objeto mudou (se vir a mesma data, rode a célula de novo). O storage não edita arquivos: apagamos o objeto antigo e gravamos um novo no mesmo lugar.

Isso é idempotência por **sobrescrita**: simples, mas o passado deixa de existir. Na versão com Spark e Iceberg os objetos **aumentam** a cada reexecução, porque o Iceberg registra uma nova versão da tabela e mantém os arquivos antigos, o que permite voltar no tempo.

**Pergunta:** se a ingestão de hoje tivesse gravado um lote com erro, como você recuperaria o lote de ontem aqui? E com Iceberg?

## Qualidade e governança neste notebook
- **Qualidade:** a Bronze é a prova do que a origem enviou, por isso não se limpa. Se a regra de limpeza estiver errada, reprocessa-se a partir daqui.
- **Linhagem:** pegue um `pedido_id` e responda de qual arquivo e de qual lote ele veio (`_source_file`, `_batch_id`).
- **Acesso:** a landing e o warehouse usam a mesma credencial `admin`. Quem deveria poder gravar na landing? Quem deveria só ler a Bronze?
- **Armazenamento x processamento:** pare o MinIO (`docker compose stop`) e rode o notebook: o pandas continua instalado, mas não há onde ler nem gravar. O dado vive no storage, o código só passa por ele.
- **Segredos:** a senha vem de variável de ambiente (`MINIO_PASSWORD`), com o padrão de laboratório. Em produção ela ficaria num cofre de segredos.

## Checkpoint
- [x] Os 4 arquivos estão em `s3://landing/mercadonordeste/lote1/`.
- [x] Há um Parquet por tabela em `s3://warehouse/bronze/<tabela>/lote1/`.
- [x] Reexecutar não duplicou, e você explica por que o número de objetos não mudou.